In [1]:
import sys

import modelopt.torch.quantization as mtq
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from modelopt.torch.export import export_hf_checkpoint

/usr/bin/x86_64-linux-gnu-ld.bfd: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/x86_64-linux-gnu-ld.bfd: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
W1004 16:35:37.431000 345615 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1004 16:35:37.446000 345615 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


In [2]:
model_path = snapshot_download("Cloudflare/clef-flash")
sys.path.insert(0, model_path)
from joint_schema_model import collate_records, encode_record, load_release_model

model, processor = load_release_model(model_path, device="cuda")
tokenizer = processor.tokenizer

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

In [3]:
model_path

'/home/dima/.cache/huggingface/hub/models--Cloudflare--clef-flash/snapshots/17f0b0ad64efb65d273590632833508766b2aae6'

In [4]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").shuffle(seed=42).select(range(10_000))

In [5]:
dataset[42]

{'prompt': "Can you summarize the credit options available for Irelanders with bad credit history?\nGenerate according to: What Are the Credit Options for Irelanders with Bad Credit History?\nHaving a bad credit score can frustrate your investment plans because you will have limited options for borrowing. There are very few, if any, banking or established financial institution that will be willing to lend you money. All is not lost because there are considerable options that Irelanders can utilize to get financing for their investments. Some of the feasible sources for borrowing include: E-payday Loans E-payday is a reliable lending institute based in Ireland. This institution focuses on providing both secure and unsecure loans to potential borrowers. You don't need to have an excellent credit history to get a loan from E-payday; they offer friendly terms to all persons who are looking to invest. Once E-payday agents determine that you can pay the loan when you receive the next salary,

In [6]:
dataset = dataset.map(
    lambda example: tokenizer(
        tokenizer.apply_chat_template(example["messages"], tokenize=False),
        truncation=True,
        max_length=2048,
        add_special_tokens=False,
    ),
    remove_columns=dataset.column_names,
)

In [7]:
print(dataset[42].keys())
print(dataset[42]["input_ids"][:50])

dict_keys(['input_ids', 'attention_mask'])
[248045, 846, 198, 6503, 488, 59987, 279, 6459, 2519, 2420, 364, 14227, 382, 440, 3743, 6459, 3712, 30, 198, 30097, 3956, 310, 25, 3437, 8448, 279, 15788, 14147, 364, 14227, 382, 440, 11132, 15788, 10776, 30, 198, 27125, 264, 3743, 6459, 5284, 628, 14042, 344, 678, 8889, 6440, 1521, 488]


In [8]:
# forward_loop runs one forward pass per sample through the text decoder, 10,000 in total. 
# Its only job is to let ModelOpt observe the activations.
def forward_loop(backbone):
    with torch.no_grad():
        for example in dataset:
            input_ids = torch.tensor([example["input_ids"]], device="cuda")
            backbone.model.language_model(input_ids=input_ids, use_cache=False)

In [9]:
%%time
# NVFP4_DEFAULT_CFG is W4A4 preset
# Quantizes both the weights and the activations to NVFP4
mtq.quantize(model.language_model, mtq.NVFP4_DEFAULT_CFG, forward_loop)

Registered <class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5VisionAttention'> to _QuantAttention for KV Cache quantization
Registered <class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5Attention'> to _QuantAttention for KV Cache quantization
Inserted 1408 quantizers
CPU times: user 36min 7s, sys: 1.35 s, total: 36min 8s
Wall time: 36min 6s


Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): QuantConv3d(
          3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16)
          (input_quantizer): TensorQuantizer(disabled)
          (output_quantizer): TensorQuantizer(disabled)
          (weight_quantizer): TensorQuantizer(disabled)
        )
      )
      (pos_embed): QuantEmbedding(
        2304, 1152
        (weight_quantizer): TensorQuantizer(disabled)
        (input_quantizer): HardDisabledTensorQuantizer(disabled)
        (output_quantizer): TensorQuantizer(disabled)
      )
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3_5VisionBlock(
          (norm1): QuantLayerNorm(
            (1152,), eps=1e-06, elementwise_affine=True, bias=True
            (input_quantizer): TensorQuantizer(disabled)
            (output_quantizer): TensorQuantizer(disabled)
         

In [10]:
with torch.inference_mode():
    export_hf_checkpoint(model.language_model, export_dir="out/clef-flash-nvfp4")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]